# 05 — Segmentation Training

Training organised **by architecture**, each with the four input scenarios as
subsections. Shared transparent loop reusing the production building blocks in
`stages/training/train_segmentation.py`.

- **Architecture 1** — DeepLabV3+CBAM (ResNet-50 encoder)
- **Architecture 2** — SegFormer (MiT-B2 encoder)
- Scenarios: `single_date`, `mt_ndvi`, `gsi`, `rf`
- Normalization `percentile` (main), loss `dynamic_balanced` (DECB-CE, main)

> Compact demo — omits production extras (preload cache, augmentation, class-balanced
> sampler, LR schedule/early-stop, MLflow, checkpoints, viz, NDVI). **GPU recommended.**

In [ ]:
# Register this repo as `cropmap_pipeline` regardless of checkout dir name.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"

_pkg = "cropmap_pipeline"
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from cropmap_pipeline import config as C

print("Repo   :", REPO)
print("Classes:", C.NUM_CLASSES, "| crops:", list(C.CDL_CLASS_NAMES.values()))

## Setup — data, band map, class weights, training helper

In [ ]:
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, Subset

from cropmap_pipeline.stages.training.train_segmentation import (
    build_model,
    compute_class_weights,
    evaluate_test_set,
    NormalizedDataset,
    _filter_s2_by_band_indices,
)
from cropmap_pipeline.stages.training.experiments.base import build_local_band_map
from cropmap_pipeline.stages.training.experiments.single_date import build_single_date_indices
from cropmap_pipeline.stages.training.experiments.mt_ndvi import build_naive_multitemporal_indices
from cropmap_pipeline.stages.training.experiments.feature_selection import build_direct_indices
from cropmap_pipeline.stages.data.spatial_split import _block_spatial_split
from cropmap_pipeline.stages.training.normalization import load_or_compute_norm_stats
from cropmap_pipeline.stages.training.losses import (
    build_wce,
    build_dynamic_balanced,
    build_focal_tversky,
)
from cropmap_pipeline.stages.selection.band_scoring import get_train_year_inputs
from geoai.geoai.train import RasterPatchDataset

NORM, LOSS = "percentile", "dynamic_balanced"  # main config
THRESH, EPOCHS = 0.5, 3  # EPOCHS: smoke; full = C.MAX_EPOCHS
DEVICE = (
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)

_yr, s2_paths, cdl_path = get_train_year_inputs()  # valid-filtered S2 + CDL (v6.1 = 2024)
names, band_to_idx, date_to_idx, mmdd_to_date = build_local_band_map(s2_paths)
cw, counts = compute_class_weights(return_counts=True)
results = {}
print(f"{len(s2_paths)} S2 dates | device {DEVICE} | norm={NORM} loss={LOSS}")


def train_and_eval(idx, tag, arch):
    """Build dataset + block split from channel indices, train `arch`, return test metrics."""
    s2_filt, local_idx = _filter_s2_by_band_indices(s2_paths, idx)
    ds_raw = RasterPatchDataset(
        s2_paths=s2_filt,
        cdl_path=cdl_path,
        patch_size=C.PATCH_SIZE,
        stride=C.STRIDE,
        keep_classes=C.KEEP_CLASSES,
        remap_lut=C.REMAP_LUT,
        min_valid_frac=C.MIN_VALID_FRAC,
        band_indices=local_idx,
    )
    band_pct = load_or_compute_norm_stats(NORM, s2_filt, Path(s2_filt[0]).parent)
    ds = NormalizedDataset(ds_raw, band_percentiles=band_pct, norm_mode=NORM)
    tr, va, te, _ = _block_spatial_split(
        [ds_raw],
        C.BLOCK_SIZE,
        C.VAL_FRAC,
        C.TEST_FRAC,
        C.NUM_CLASSES,
        C.SEED,
        min_class_frac=C.MIN_CLASS_FRAC,
    )
    mk = lambda i, sh: DataLoader(
        Subset(ds, i), batch_size=C.BATCH_SIZE, shuffle=sh, num_workers=2, drop_last=sh
    )
    train_dl, val_dl = mk(tr, True), mk(va, False)
    test_dl = mk(te, False) if te else None

    model = build_model(arch, len(local_idx), C.NUM_CLASSES).to(DEVICE)
    criterion = (
        build_wce(cw.to(DEVICE))
        if LOSS == "wce"
        else (
            build_focal_tversky(class_counts=counts)
            if LOSS == "focal_tversky"
            else build_dynamic_balanced(num_classes=C.NUM_CLASSES)
        )
    )
    if hasattr(criterion, "to"):
        criterion = criterion.to(DEVICE)
    cfg = C.ARCH_CFG[arch]
    optimizer = (
        torch.optim.SGD(
            model.parameters(), lr=cfg["lr"], momentum=0.9, weight_decay=cfg["weight_decay"]
        )
        if cfg["optimizer"] == "sgd"
        else torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    )
    print(f"[{arch} | {tag}] {len(local_idx)} ch | {len(tr)}/{len(va)}/{len(te)} patches")
    val = None
    for ep in range(1, EPOCHS + 1):
        model.train()
        run = 0.0
        for imgs, masks in train_dl:
            imgs = torch.nan_to_num(imgs).to(DEVICE)
            masks = masks.to(DEVICE).long()
            optimizer.zero_grad()
            l = criterion(model(imgs), masks)
            l.backward()
            optimizer.step()
            run += l.item()
        val = evaluate_test_set(model, val_dl, C.NUM_CLASSES, DEVICE)
        print(f'  ep{ep:>2}: train_loss {run/len(train_dl):.4f} | val mIoU {val["miou"]:.4f}')
    res = evaluate_test_set(model, test_dl, C.NUM_CLASSES, DEVICE) if test_dl else val
    print(f'  TEST mIoU {res["miou"]:.4f}  mF1 {res["mf1"]:.4f}  OA {res["oa"]:.4f}')
    return res

## Build channels (shared across both architectures)

Channel sets depend only on the scenario, not the architecture — build once, reuse.

In [ ]:
CHANNELS = {}
CHANNELS["single_date"], _, _sd_date = build_single_date_indices(
    date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path
)
CHANNELS["mt_ndvi"], _, _phenol = build_naive_multitemporal_indices(
    date_to_idx, band_to_idx, s2_paths=s2_paths, cdl_path=cdl_path
)
CHANNELS["gsi"], _ = build_direct_indices(
    C.PROCESSED_DIR / f"select_gsi_direct_s{THRESH:g}.json",
    mmdd_to_date,
    band_to_idx,
    selector_name="gsi",
    subset_k=None,
)
CHANNELS["rf"], _ = build_direct_indices(
    C.PROCESSED_DIR / f"select_rf_direct_s{THRESH:g}.json",
    mmdd_to_date,
    band_to_idx,
    selector_name="rf",
    subset_k=None,
)
print("peak-NDVI date:", _sd_date, "| quarterly:", _phenol)
for k, v in CHANNELS.items():
    print(f"  {k:12s}: {len(v)} channels")

# Architecture 1 — DeepLabV3+CBAM (ResNet-50)

DeepLabV3+ with CBAM attention, ResNet-50 encoder. Trained on each scenario below.

### single_date — peak-NDVI date × all bands (baseline)

Peak-NDVI date over crop pixels; all 10 bands, no selection.

In [ ]:
results[("deeplabv3plus_cbam", "single_date")] = train_and_eval(
    CHANNELS["single_date"], "single_date", arch="deeplabv3plus_cbam"
)

### mt_ndvi — 4 per-quarter peak-NDVI dates × bands

One max-NDVI date per calendar quarter (multi-temporal baseline).

In [ ]:
results[("deeplabv3plus_cbam", "mt_ndvi")] = train_and_eval(
    CHANNELS["mt_ndvi"], "mt_ndvi", arch="deeplabv3plus_cbam"
)

### gsi — GSI selection (normalized score ≥ 0.5)

Channels from `select_gsi_direct_s0.5.json` (notebook 04).

In [ ]:
results[("deeplabv3plus_cbam", "gsi")] = train_and_eval(
    CHANNELS["gsi"], "gsi", arch="deeplabv3plus_cbam"
)

### rf — RF-importance selection (normalized score ≥ 0.5)

Channels from `select_rf_direct_s0.5.json` (notebook 04).

In [ ]:
results[("deeplabv3plus_cbam", "rf")] = train_and_eval(
    CHANNELS["rf"], "rf", arch="deeplabv3plus_cbam"
)

# Architecture 2 — SegFormer (MiT-B2)

SegFormer transformer segmentation head, MiT-B2 encoder. Trained on each scenario below.

### single_date — peak-NDVI date × all bands (baseline)

Peak-NDVI date over crop pixels; all 10 bands, no selection.

In [ ]:
results[("segformer", "single_date")] = train_and_eval(
    CHANNELS["single_date"], "single_date", arch="segformer"
)

### mt_ndvi — 4 per-quarter peak-NDVI dates × bands

One max-NDVI date per calendar quarter (multi-temporal baseline).

In [ ]:
results[("segformer", "mt_ndvi")] = train_and_eval(CHANNELS["mt_ndvi"], "mt_ndvi", arch="segformer")

### gsi — GSI selection (normalized score ≥ 0.5)

Channels from `select_gsi_direct_s0.5.json` (notebook 04).

In [ ]:
results[("segformer", "gsi")] = train_and_eval(CHANNELS["gsi"], "gsi", arch="segformer")

### rf — RF-importance selection (normalized score ≥ 0.5)

Channels from `select_rf_direct_s0.5.json` (notebook 04).

In [ ]:
results[("segformer", "rf")] = train_and_eval(CHANNELS["rf"], "rf", arch="segformer")

## Compare — architecture × scenario

In [ ]:
rows = {
    f"{arch} / {tag}": {m: r[m] for m in ("miou", "mf1", "oa")}
    for (arch, tag), r in results.items()
}
df = pd.DataFrame(rows).T.round(4)
print(df)
if len(df):
    import matplotlib.pyplot as plt

    df["miou"].plot.bar(color="steelblue", figsize=(9, 4))
    plt.ylabel("test mIoU")
    plt.title("Architecture × scenario")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()